## Introduction

To get you familiar with the setup for the regression challenge, we have prepared this notebook for you. Before you start here read the `challenge_description.md` 


### Data
Attached in this homework you will find data equivalent in format to the data from Timo's Regression Challenge.

1) `labeled_data.csv`: this is your training dataset
2) `unlabeled_data.csv`: this is your test data, it does not include the column "Quantity_Sold_(kilo)" because you need to create/predict this column with the model you train


This notebook will create a file "prediction_solution.csv" that you can upload to https://redi-ml-regression-challenge.streamlit.app/ (the challenge leaderboard) to see your MSE (mean squared error) test score.

### Feature Encoding & Modeling

Train at least 3 different models/use different features.

Your task for this homework intro is to 
1) change at least one thing about the features (in addition to simple encoding) and 
2) apply at least one regularization technique.
3) Choose your best model and uplaod one prediction to the scoreboard.

## Imports

Selfnote ignore it; This code splits the dataset into a training set and a validation set. The training set will be used to teach the model, while the validation set will be used to evaluate how well the model performs before creating final predictions. This helps us avoid overfitting and provides a realistic estimate of model performance.

In [20]:
import pandas as pd

labeled_df = pd.read_csv("data/labeled_data.csv")

unlabeled_df = pd.read_csv("data/unlabeled_data.csv")

example_df = pd.read_csv("data/prediction_example.csv")

## Loading Data

### You can ignore this if you already know this: 
this is just a self-note. This code displays the first five rows of the dataset. 
I use it to understand the available columns and get a 
quick overview of the data before preprocessing and model building.

In [21]:
# Change this to the full data path `labeled_data.csv` from Timo's challenge to load the full data and see all available columns instead.
full_data_df = pd.read_csv("data/labeled_data.csv")
full_data_df.head()

,ID,Item Code,Quantity_Sold_(kilo),Unit Selling Price (RMB/kg),Item Name_x,Category Code,Category Name,Wholesale Price (RMB/kg),Item Name_y,Loss Rate (%),Weekday
0,1,102900005115762,6.841,6.00,Amaranth,1011010101,Flower/Leaf Vegetables,3.88,Amaranth,18.52,Wednesday
1,21,102900005117209,1.909,16.00,7 Colour Pepper (1),1011010504,Capsicum,7.58,7 Colour Pepper (1),15.98,Wednesday
2,22,102900005118817,5.472,14.00,Spinach,1011010101,Flower/Leaf Vegetables,8.47,Spinach,18.51,Wednesday
3,23,102900005118824,4.119,10.00,High Melon (1),1011010402,Aquatic Tuberous Vegetables,5.65,High Melon (1),29.25,Wednesday
4,24,102900005118831,10.000,6.85,Wawacai,1011010101,Flower/Leaf Vegetables,4.13,Wawacai,2.48,Wednesday


### You can ignore this if you already know this: 
this is just a self-note. This code counts missing values in each column.
I use it to identify which variables require cleaning 
or imputation before training machine learning models.

In [22]:
full_data_df.isnull().sum()

ID                                0
Item Code                         0
Quantity_Sold_(kilo)              0
Unit Selling Price (RMB/kg)       0
Item Name_x                       0
Category Code                     0
Category Name                     0
Wholesale Price (RMB/kg)       1678
Item Name_y                       0
Loss Rate (%)                     0
Weekday                           0
dtype: int64

## Split into Train/Val(/Test)

### Selfnote ignore it; This code splits the dataset into 
training and validation datasets. The training dataset will be used to teach the model, while the validation dataset will be used to evaluate 
the model before creating final predictions. Using separate validation data
helps us measure model performance on unseen data and avoid overfitting.

In [23]:
X = full_data_df.drop(columns=["Quantity_Sold_(kilo)"])
y = full_data_df["Quantity_Sold_(kilo)"]

from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

## Explore the training data

Timo maybe changed this data up a bit from the initial EDA homework in Week 1.

Apply what you have learned about EDA and look into the features

### Selfnote ignore it; This code explores the training dataset after splitting. It shows the available features,
identifies missing values, and provides an overview of the first rows. 
This helps me understand the data before applying feature engineering and preprocessing.

In [24]:
X_train.info()

X_train.isnull().sum()

X_train.head()

<class 'pandas.DataFrame'>
Index: 24464 entries, 12273 to 23654
Data columns (total 10 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ID                           24464 non-null  int64  
 1   Item Code                    24464 non-null  int64  
 2   Unit Selling Price (RMB/kg)  24464 non-null  float64
 3   Item Name_x                  24464 non-null  str    
 4   Category Code                24464 non-null  int64  
 5   Category Name                24464 non-null  str    
 6   Wholesale Price (RMB/kg)     23207 non-null  float64
 7   Item Name_y                  24464 non-null  str    
 8   Loss Rate (%)                24464 non-null  float64
 9   Weekday                      24464 non-null  str    
dtypes: float64(3), int64(3), str(4)
memory usage: 2.1 MB


,ID,Item Code,Unit Selling Price (RMB/kg),Item Name_x,Category Code,Category Name,Wholesale Price (RMB/kg),Item Name_y,Loss Rate (%),Weekday
12273,12295,102900005119944,12.0,Haixian Mushroom (1),1011010801,Edible Mushroom,6.87,Haixian Mushroom (1),9.89,Tuesday
16782,16806,102900051010455,6.0,Huangbaicai (2),1011010101,Flower/Leaf Vegetables,3.12,Huangbaicai (2),15.61,Tuesday
16773,16792,102900005125815,8.0,Apricot Bao Mushroom (1),1011010801,Edible Mushroom,4.27,Apricot Bao Mushroom (1),5.05,Tuesday
31923,31899,102900005116219,12.0,Red Hot Peppers,1011010504,Capsicum,5.30,Red Hot Peppers,6.73,Saturday
8723,8697,102900005115779,12.0,Yunnan Shengcai,1010101,Flower/Leaf Vegetables,8.06,Yunnan Shengcai,15.25,Monday


### Selfnote ignore it; This code counts missing values in each feature of
the training dataset. I use it to identify which variables require cleaning before training the model.
The output shows that Wholesale Price (RMB/kg) contains missing values and therefore needs imputation. -->

In [25]:

X_train.isnull().sum()


ID                                0
Item Code                         0
Unit Selling Price (RMB/kg)       0
Item Name_x                       0
Category Code                     0
Category Name                     0
Wholesale Price (RMB/kg)       1257
Item Name_y                       0
Loss Rate (%)                     0
Weekday                           0
dtype: int64

In [26]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 24464 entries, 12273 to 23654
Data columns (total 10 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ID                           24464 non-null  int64  
 1   Item Code                    24464 non-null  int64  
 2   Unit Selling Price (RMB/kg)  24464 non-null  float64
 3   Item Name_x                  24464 non-null  str    
 4   Category Code                24464 non-null  int64  
 5   Category Name                24464 non-null  str    
 6   Wholesale Price (RMB/kg)     23207 non-null  float64
 7   Item Name_y                  24464 non-null  str    
 8   Loss Rate (%)                24464 non-null  float64
 9   Weekday                      24464 non-null  str    
dtypes: float64(3), int64(3), str(4)
memory usage: 2.1 MB


## Encode Features & Create new Features

### Selfnote ignore it; This code prepares the dataset for machine learning. 
Numerical columns are cleaned using median imputation to handle missing values, while categorical
columns are cleaned using the most frequent category and converted into numerical
values using One-Hot Encoding because machine learning models cannot work directly with text.

In [27]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_features = [
    "Unit Selling Price (RMB/kg)",
    "Wholesale Price (RMB/kg)",
    "Loss Rate (%)",
    "Category Code"
]

categorical_features = [
    "Category Name",
    "Weekday"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numeric_features
        ),
        (
            "cat",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "encoder",
                    OneHotEncoder(handle_unknown="ignore")
                )
            ]),
            categorical_features
        )
    ]
)

## Create a model and train it

### Selfnote ignore it; This code creates a Linear Regression model and combines it with the preprocessing 
pipeline. The model learns the relationship between 
the selected features and the target variable Quantity_Sold_(kilo) using the training dataset.

In [28]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['ID','Item Code','Unit Selling Price (RMB/kg)',...,'Item Name_y', 'Loss Rate (%)','Weekday']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed throu

### Selfnote ignore it; This code creates a reusable function called report(). 
The function automatically calculates MAE, MSE, and R² and prints them
in a clean format. I use it because 
I will compare several machine learning models and want a simple way to evaluate each one.

In [31]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def report(y_true, y_pred, label):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(
        f"{label:20s} | MAE {mae:.3f} | MSE {mse:.3f} | R² {r2:.3f}"
    )

## Evaluate the model

### Selfnote ignore it; This code evaluates the Linear Regression model using validation data. It compares the predicted values with the real values and reports MAE, MSE, and R². These metrics help me measure the model's performance and compare it with other models later.

In [32]:
print("Validation Error:")

report(
    y_val,
    model.predict(X_val),
    "Linear Regression"
)

Validation Error:
Linear Regression    | MAE 7.650 | MSE 145.875 | R² -0.000


Selfnote ignore it; The first Linear Regression model achieved 
a MAE of 7.650, an MSE of 145.875, and an R² score close to zero. This indicates that
the selected features do not explain the target variable very well,
and a more advanced model or additional feature engineering is needed.

## Repeat at least twice

Train at least 3 different models/use different features.

Your task for this homework intro is to 
1) change at least one thing about the features (in addition to simple encoding) and 
2) apply at least one regularization technique.

So you should have at least 3 different validation errors.

3) choose the model with the best score and uplaod your prediction to the [scoreboad](https://s26-regression-challenge.streamlit.app/).

## Be as creative as you want to try other models, play around with features etc.

## Repeat at least twice.

### Selfnote ignore it; This code creates a Ridge Regression model, which is
a regularized version of Linear Regression. 
Regularization helps reduce overfitting
by penalizing large coefficients and can improve generalization on unseen data.

In [33]:
from sklearn.linear_model import Ridge

ridge_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Ridge(alpha=1.0))
])

ridge_model.fit(X_train, y_train)

/Users/hamidi/CODING_LIFE/A1_Education/REDI_school/Machien_lerning/ml-course/f26-redi-ml-hamidi015/.venv/lib/python3.15/site-packages/sklearn/linear_model/_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 4.483891343639119e-22.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['ID','Item Code','Unit Selling Price (RMB/kg)',...,'Item Name_y', 'Loss Rate (%)','Weekday']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed throu

## Evaluate Ridge Regression

### Selfnote ignore it; The Ridge Regression model performed
better than the Linear Regression model. The MAE and MSE values
decreased, while the R² score increased from approximately 
0.00 to 0.11. This suggests that regularization helped the
model generalize better and improved its predictive performance.

In [34]:
print("Validation Error:")

report(
    y_val,
    ridge_model.predict(X_val),
    "Ridge Regression"
)

Validation Error:
Ridge Regression     | MAE 6.948 | MSE 129.610 | R² 0.111


### Selfnote ignore it; This code creates a Random Forest Regression model. Random Forest combines many decision
trees and can learn more complex patterns in the data than linear models.
It often performs better when relationships between variables are non-linear.

In [35]:
from sklearn.ensemble import RandomForestRegressor

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "regressor",
        RandomForestRegressor(
            n_estimators=100,
            random_state=42
        )
    )
])

rf_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['ID','Item Code','Unit Selling Price (RMB/kg)',...,'Item Name_y', 'Loss Rate (%)','Weekday']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed throu

### Selfnote ignore it; This code evaluates the Random Forest model on the
validation dataset. The results will be compared with Linear Regression
and Ridge Regression to determine which model performs best.

In [36]:
print("Validation Error:")

report(
    y_val,
    rf_model.predict(X_val),
    "Random Forest"
)

Validation Error:
Random Forest        | MAE 4.294 | MSE 65.599 | R² 0.550


In [38]:
rf_model.predict(X_val)

array([ 0.75312183,  6.39321   ,  6.35397   , ..., 10.77324   ,
        6.09032   ,  4.18234   ], shape=(8155,))

In [40]:
sarahs_test_data = pd.read_csv("data/unlabeled_data.csv")
# This is equivalent to just X_test, Timo's website will evaluate your predictions on the hidden test set (y_test).

# TODO: Apply the same feature engineering to the test data as you did to the training data.
# sarahs_test_data_cleaned = apply_feature_engineering(sarahs_test_data)
# Note: You should not drop any rows from this test data, otherwise your submission will have missing rows and won't work.
# Make sure to not change the order of the rows either, that would mess up the IDs.

# TODO: Create your predictions for the test data.
# your_test_predictions = model.predict(sarahs_test_data_cleaned)

prediction_IDs = pd.read_csv("data/prediction_example.csv")["ID"]

# TODO: Create your prediction_solution.csv to upload, just uncomment this code part after you created your_test_predictions.
# prediction_solution = pd.DataFrame(
#    {"ID": prediction_IDs, "Quantity_Sold_(kilo)": your_test_predictions}
# )

# prediction_solution.to_csv(
#     "prediction_solution.csv", index=False
# )

### Selfnote ignore it;
I load the unlabeled dataset because it contains the rows for which I need to
predict Quantity_Sold_(kilo). This dataset will be used for the final challenge submission.

In [41]:
your_test_predictions = rf_model.predict(
    sarahs_test_data
)

### Selfnote ignore it;

I use the Random Forest model because it achieved the best
validation performance. In this step the model generates predictions for every row in the unlabeled dataset.

In [42]:
your_test_predictions = rf_model.predict(
    sarahs_test_data
)

If you have any questions read the `challenge_description.md`. If you still have questions ask in slack group chat.

Have fun and help each other learn! :)

### Selfnote ignore it;
I create a new dataframe that contains the original
IDs and the predictions from my Random Forest model. 
This dataframe will become the final submission file for the challenge.

In [43]:
prediction_solution = pd.DataFrame(
    {
        "ID": prediction_IDs,
        "Quantity_Sold_(kilo)": your_test_predictions
    }
)

### Selfnote ignore it;

I check the first rows of the prediction file to make sure the structure looks correct before saving it.

In [44]:
prediction_solution.head()

,ID,Quantity_Sold_(kilo)
0,32650,6.995000
1,32643,5.655833
2,32644,5.540000
3,32645,14.052333
4,32646,3.452470


### Selfnote ignore it;

I save the final prediction file in CSV format. This file contains 
the IDs and the predicted values and will be uploaded to the challenge leaderboard.

In [47]:
prediction_solution.to_csv(
    "prediction_solution.csv",
    index=False
)


### Selfnote ignore it;

I check the files in my current folder to confirm that prediction_solution.csv was created successfully.

In [46]:
import os

os.listdir()

['.DS_Store',
 'homework_regression_challenge_intro.ipynb',
 'challenge_description.md',
 'data',
 'prediction_solution.csv']

### Selfnote ignore it;

I save the final prediction file as a CSV file. 
This file contains the IDs and the predicted 
Quantity_Sold_(kilo) values and will be uploaded to the challenge leaderboard.

In [49]:
prediction_solution.to_csv(
    "prediction_solution.csv",
    index=False
)

In [50]:
import os

os.listdir()

['.DS_Store',
 'homework_regression_challenge_intro.ipynb',
 'challenge_description.md',
 'data',
 'prediction_solution.csv']

Selfnote ignore it;

In this project, I explored the dataset, handled missing values, encoded categorical features, 
and trained three different machine learning models. Linear Regression was used as a baseline model, 
Ridge Regression was used to apply regularization, and Random Forest was used as a more advanced model. 
After comparing the validation results, Random Forest achieved the best performance
with the lowest errors and the highest R² score. Therefore, I selected Random Forest as the final model and used
it to generate predictions for the unlabeled dataset. The predictions were saved in the required submission format.

## Challenges and Lessons Learned
During this project I faced a few challenges.

The first challenge was understanding how to preprocess numerical and categorical features correctly. At the beginning I was unsure which columns should be encoded and which columns should be treated as numerical features. After reviewing the course materials and discussing the problem with classmates and AI tools, I learned how to use One-Hot Encoding and imputation correctly.

Another challenge was understanding the difference between training data, validation data, and the unlabeled challenge dataset. At first I confused these datasets, but after experimenting with the code and reading the notebook instructions, I better understood the machine learning workflow.

I also faced some issues while generating the final prediction file. I received errors related to missing variables and data loading. By checking the error messages, reviewing my code, and asking for help, I was able to solve these problems and successfully create the final prediction_solution.csv file.

Overall, this project helped me understand preprocessing, model evaluation, regularization, and how to compare different machine learning models before selecting the best one.